[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GeomaticsCaminosUPM/footprint_attributes/blob/main/examples/run.ipynb)

In [1]:
import sys

# Colab has neither this package nor its git-hosted map-plotting dependency
# (FancyFolium) preinstalled -- install both, pinned to the v1.0.0 release,
# only when actually running in Colab. A local checkout should instead run
# `uv sync --all-groups` once (installs the `visualization` dependency
# group) rather than pip-installing into whatever environment this kernel
# happens to use.
if "google.colab" in sys.modules:
    %pip install -q "footprint-attributes @ git+https://github.com/GeomaticsCaminosUPM/footprint_attributes.git@v1.0.0"
    %pip install -q "FancyFolium @ git+https://github.com/GeomaticsCaminosUPM/FancyFolium.git@v1.0.0" folium mapclassify

# `footprint_attributes.run()` — the single entry point

The other example notebooks (`direction`, `shape`, `position`,
`building_sizes`) each exercise one module's functions directly. In
practice you usually just want "these columns, on this GeoDataFrame",
without caring which module computes what -- that's what
`footprint_attributes.run()` is for: it wires `direction`/`shape`/`position`
together behind a single call, driven by a `config` dict.

```python
result = footprint_attributes.run(
    footprints_gdf,
    config={
        "columns": [...],       # required: exact column names, norm names
                                 # ("EC8", "ASCE7", ...), "position", "bearing",
                                 # or "slenderness"
        "position": {...},      # optional: kwargs forwarded to position()
        "shape": {...},         # optional: kwargs forwarded to shape()
        "direction": {...},     # optional: kwargs forwarded to direction.inertia()
    },
)
```

This notebook runs it on the three real pilot-area datasets (San José,
Guatemala City, Santo Domingo), requesting a representative mix of columns
-- a couple of full norms, the position pipeline, and two code-independent
indices -- and shows the result on one merged map.

## Setup

In [2]:
import os

import geopandas as gpd
import pandas as pd

import footprint_attributes
from footprint_attributes.notebook_utils import load_city

# ─────────────────────────────────────────────────────────────────────────
# Configuration — edit this block to point the notebook at different data.
# Everything below only depends on these variables, so swapping in your
# own footprints/region names requires no other changes.
# ─────────────────────────────────────────────────────────────────────────
CITY_FILES = {
    "san_jose": "data/san_jose_pilot_region.gpkg",
    "guatemala": "data/guatemala_pilot_region.gpkg",
    "santo_domingo": "data/santo_domingo_pilot_region.gpkg",
}
# Name of a column with building heights (metres), if your data has one.
# The sample pilot-region files ship geometry only, so this is left as
# None -- position()/run() then treat every building as height = 1. If you
# have real heights, set this and pass it via RUN_CONFIG["position"] below.
HEIGHT_COLUMN = None

In [3]:
output_path = "data/output"

os.makedirs(output_path, exist_ok=True)

## A single `run()` call

One config dict requests: the full `EC8` and `GNDTII` norms (each expands
to that code's own parameter columns, plus a `compliance_*` 0-100 score per
parameter), the `position` pipeline (5 columns, computed atomically), the
`bearing` (from `direction.inertia`), and two code-independent compactness
indices. `position`'s `buffer`/`height_column` are forwarded via the
`"position"` sub-dict -- `HEIGHT_COLUMN` is `None` here since the sample
pilot-region files carry geometry only, so every building is treated as
height = 1 (see the config cell in Setup for how to point this at a real
height column).

In [4]:
RUN_CONFIG = {
    "columns": [
        "EC8", "GNDTII", "position", "bearing",
        "polsby_popper", "convex_hull_irregularity", 
        "NTC23_setbackRatio"
    ],
    "position": {"buffer": 0.1, "height_column": HEIGHT_COLUMN},
}

san_jose_demo = footprint_attributes.run(load_city(CITY_FILES["san_jose"], fill_height_column=HEIGHT_COLUMN), config=RUN_CONFIG)
new_columns = [c for c in san_jose_demo.columns if c != "geometry"]
print(f"{len(new_columns)} columns computed:")
print(new_columns)
san_jose_demo[["relativePosition", "bearing", "EC8_compactness", "GNDTII_beta2_setbackRatio", "NTC23_setbackRatio"]].head()

28 columns computed:
['id', 'dataset', 'city', 'height', 'EC8_compactness', 'GNDTII_beta6_setbackSlenderness', 'compliance_GNDTII_beta4_eccentricityRatio', 'compliance_EC8_eccentricityRatio', 'bearing', 'NTC23_setbackRatio', 'GNDTII_beta2_setbackRatio', 'compliance_EC8_radiusRatio', 'polsby_popper', 'GNDTII_beta4_eccentricityRatio', 'compliance_EC8_compactness', 'compliance_GNDTII_beta1_mainShapeSlenderness', 'GNDTII_beta1_mainShapeSlenderness', 'compliance_GNDTII_beta2_setbackRatio', 'EC8_radiusRatio', 'convex_hull_irregularity', 'compliance_GNDTII_beta6_setbackSlenderness', 'EC8_eccentricityRatio', 'contact_force', 'contact_confinementRatio', 'contact_angularAcc', 'contact_angle', 'contact_height', 'relativePosition']


,relativePosition,bearing,EC8_compactness,GNDTII_beta2_setbackRatio,NTC23_setbackRatio
0,confined,14.395737,1.000000,0.000000,0.000000
1,corner,3.377607,0.999820,0.000000,0.000000
2,confined,24.013840,0.999880,0.000000,0.000000
3,confined,-85.975534,0.997747,0.004441,0.004441
4,isolated,-79.978056,0.835991,0.338724,0.338724


## Run on all three pilot areas

In [5]:
results = {}
for city, path in CITY_FILES.items():
    gdf = load_city(path, fill_height_column=HEIGHT_COLUMN)
    results[city] = footprint_attributes.run(gdf, config=RUN_CONFIG)
    print(f"{city:15s} {len(results[city]):5d} footprints, {len(results[city].columns)} columns")

san_jose          911 footprints, 29 columns


guatemala         436 footprints, 29 columns


santo_domingo    1464 footprints, 29 columns


## Result

One GeoDataFrame per city, each carrying every column requested above,
saved to `data/output/` and shown on a single merged map (one tab per
city, same layer set in each):

- `relativePosition` -- categorical position class.
- `bearing` -- building orientation (deg from North, range [-90, 90]).
- `EC8_compactness` -- green near 1.0 (convex), red below the 0.95 limit.
- `GNDTII_beta2_setbackRatio` -- green near 0, red above the code limit.
- `polsby_popper` -- green near 1 (circular), red near 0 (elongated).

In [6]:
san_jose_gdf = results["san_jose"]
guatemala_gdf = results["guatemala"]
santo_domingo_gdf = results["santo_domingo"]

san_jose_gdf.to_file(output_path + "/san_jose_run.gpkg")
guatemala_gdf.to_file(output_path + "/guatemala_run.gpkg")
santo_domingo_gdf.to_file(output_path + "/santo_domingo_run.gpkg")

In [7]:
map_columns = [
    "relativePosition", "bearing", "EC8_compactness",
    "GNDTII_beta2_setbackRatio", "polsby_popper",
]
columns_and_layer_names = [
    {
        "column": "relativePosition",
        "layer_name": "Relative Block Position",
        "categorical": True,
        "active": True,
    },
    {
        "column": "bearing",
        "layer_name": "Bearing (deg from North)",
        "cmap": "viridis",
        "vmin": -90,
        "vmax": 90,
        "active": False,
    },
    {
        "column": "EC8_compactness",
        "layer_name": "EC8 Compactness",
        "cmap": "RdYlGn",
        "vmin": 0.5,
        "vmax": 1.0,
        "active": False,
    },
    {
        "column": "GNDTII_beta2_setbackRatio",
        "layer_name": "GNDTII beta2 (setback ratio)",
        "cmap": "RdYlGn_r",
        "vmin": 0.0,
        "vmax": 0.4,
        "active": False,
    },
    {
        "column": "polsby_popper",
        "layer_name": "Polsby-Popper Compactness",
        "cmap": "RdYlGn",
        "vmin": 0.0,
        "vmax": 1.0,
        "active": False,
    },
]

In [8]:
from footprint_attributes.visualization import build_comparison_map

m = build_comparison_map(
    {"san_jose": san_jose_gdf, "guatemala": guatemala_gdf, "santo_domingo": santo_domingo_gdf},
    columns_and_layer_names,
    popup_columns=map_columns,
    labels={"san_jose": "San Jose", "guatemala": "Guatemala", "santo_domingo": "Santo Domingo"},
)
m.save(output_path + "/map_run.html")
m

## Interactive 3D map (`build_map` viewer)

The folium map above merges the columns *this* notebook happened to
request. `footprint_attributes.visualization.build_map` instead always
computes the *full* set (all 15 EC8/ASCE7/GNDTII/CSCR2010/NTC-23 shape
metrics, the 4-category shape index, `relativePosition`, and both
slenderness conventions) plus every geometry overlay, from the raw
footprints -- so it doesn't need `RUN_CONFIG` at all, and gives a
strictly bigger, browser-native 3D view (MapLibre + deck.gl, buildings
extruded by height, no backend) with:

- a colour legend for whichever column is active, plus a norm-exceedance
  side chart for each shape metric;
- an **Overlays** panel with all five geometry overlays at once — convex
  hull, minimum **bounding box**, inertia axis, basic-length dimension
  **arrows**, and the net contact-force resultant arrow;
- an auto-playing tour (▶, top right) that cycles through relative
  position, the shape index, and every individual shape metric.

Needs the ``visualization`` extra: ``pip install "footprint-attributes[visualization]"``. See
also ``examples/generate_interactive_map.py`` for the same call as a
standalone script.

In [9]:
from footprint_attributes.visualization import build_map

interactive_map_dir = output_path + "/interactive_map_run"
build_map(
    {
        "san_jose": load_city(CITY_FILES["san_jose"], fill_height_column=HEIGHT_COLUMN),
        "guatemala": load_city(CITY_FILES["guatemala"], fill_height_column=HEIGHT_COLUMN),
        "santo_domingo": load_city(CITY_FILES["santo_domingo"], fill_height_column=HEIGHT_COLUMN),
    },
    interactive_map_dir,
    labels={
        "san_jose": "San José — Mata Redonda",
        "guatemala": "Guatemala — Zona 10",
        "santo_domingo": "Santo Domingo — Ensanche Quisquella",
    },
    default_dataset="san_jose",
    title="Pilot regions — footprint_attributes",
)
print(f"wrote {interactive_map_dir}")
print(f"serve it with: python -m http.server --directory {interactive_map_dir}")

wrote data/output/interactive_map_run
serve it with: python -m http.server --directory data/output/interactive_map_run
